# Initialer Code

Folgender Code muss ausgeführt werden, um die Usecases zu validieren


In [1]:
endpoints = ["http://owid.de/api/fcs/"]; # Liste der Endpunkte, die abgefragt werden sollen.

from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
import xml.etree.ElementTree as ET
import requests

ns = {
    'sru': 'http://docs.oasis-open.org/ns/search-ws/sruResponse',
    'fcs': 'http://clarin.eu/fcs/resource',
    'hits': 'http://clarin.eu/fcs/dataview/hits',
    'lex': 'http://clarin.eu/fcs/dataview/lex'
}

@dataclass
class Hit:
    kind: Optional[str]
    value: str

@dataclass
class LexValue:
    text: str
    type: Optional[str] = None
    source: Optional[str] = None
    vocabValueRef: Optional[str] = None
    preferred: bool = False

@dataclass
class Record:
    pid: Optional[str]
    fragment_ref: Optional[str]
    hits: List[Hit] = field(default_factory=list)
    lex_fields: Dict[str, List[LexValue]] = field(default_factory=dict)
    record_position: Optional[int] = None

def _get_text(el):
    return el.text.strip() if el is not None and el.text is not None else None

def parse_sru(xml_text: str) -> List[Record]:
    root = ET.fromstring(xml_text)
    records: List[Record] = []
    for rec_el in root.findall('.//sru:record', ns):
        pos_el = rec_el.find('sru:recordPosition', ns)
        rec_pos = int(pos_el.text) if pos_el is not None and pos_el.text else None

        resource = rec_el.find('.//fcs:Resource', ns)
        pid = resource.get('pid') if resource is not None else None

        frag_el = rec_el.find('.//fcs:ResourceFragment', ns)
        frag_ref = frag_el.get('ref') if frag_el is not None else None

        hits_list: List[Hit] = []
        if frag_el is not None:
            for dv in frag_el.findall('fcs:DataView', ns):
                dv_type = dv.get('type','')
                if 'hits' in dv_type:
                    for hit_el in dv.findall('.//hits:Hit', ns):
                        hits_list.append(Hit(kind=hit_el.get('kind'), value=_get_text(hit_el) or ''))

        lex_fields: Dict[str, List[LexValue]] = {}
        if frag_el is not None:
            for dv in frag_el.findall('fcs:DataView', ns):
                dv_type = dv.get('type','')
                if 'lex' in dv_type:
                    entry = dv.find('lex:Entry', ns)
                    if entry is None:
                        continue
                    for field_el in entry.findall('lex:Field', ns):
                        ftype = field_el.get('type')
                        vals: List[LexValue] = []
                        for v in field_el.findall('lex:Value', ns):
                            vals.append(LexValue(
                                text=_get_text(v) or '',
                                type=v.get('type'),
                                source=v.get('source'),
                                vocabValueRef=v.get('vocabValueRef'),
                                preferred=(v.get('preferred') == 'true')
                            ))
                        if ftype:
                            lex_fields.setdefault(ftype, []).extend(vals)

        records.append(Record(pid=pid, fragment_ref=frag_ref, hits=hits_list, lex_fields=lex_fields, record_position=rec_pos))
    return records

def to_flat(record: Record) -> Dict[str, Any]:
    def first(ftype):
        vals = record.lex_fields.get(ftype)
        return vals[0].text if vals else None
    def all_texts(ftype):
        return [v.text for v in record.lex_fields.get(ftype, [])]

    landing = next((v.text for v in record.lex_fields.get('ref',[]) if v.type == 'landingpage'), None)
    return {
        'pid': record.pid,
        'ref': record.fragment_ref,
        'lemma': first('lemma'),
        'entryId': first('entryId'),
        'landingpage': landing,
        'pos': all_texts('pos'),
        'gender': all_texts('gender'),
        'segmentation': first('segmentation'),
        'definition': first('definition'),
        'citation_examples': [{'source': v.source, 'text': v.text} for v in record.lex_fields.get('citation', [])],
        'hits': [{'kind': h.kind, 'value': h.value} for h in record.hits],
        'recordPosition': record.record_position
    }

def fetch_and_parse(url: str, timeout: int = 10) -> List[Dict[str, Any]]:
    resp = requests.get(url, timeout=timeout, headers={'Accept': 'application/xml'})
    resp.raise_for_status()
    raw = parse_sru(resp.text)
    return [to_flat(r) for r in raw]

def search(query):
  for endpoint in endpoints:
    url = f"{endpoint}?queryType=lex&query={query}"; # URL für die Abfrage erstellen
    return fetch_and_parse(url)

In [2]:
data = search("Berg")